In [3]:
import pandas as pd

# On ne charge que les colonnes utiles à notre contrôle qualité
colonnes = [
    "code", "product_name", "brands", "quantity", "categories_en", "countries_en",
    "energy-kcal_100g", "fat_100g", "saturated-fat_100g", "carbohydrates_100g",
    "sugars_100g", "proteins_100g", "salt_100g",
]

morceaux = pd.read_csv(
    "data/openfoodfacts.csv.gz",
    sep="\t",                  # les colonnes sont séparées par des tabulations
    usecols=colonnes,
    dtype={"code": str},       # le code-barres est un texte, pas un nombre (sinon on perd les zéros du début)
    chunksize=100_000,         # lecture par morceaux de 100 000 lignes
    on_bad_lines="skip",       # on ignore les rares lignes mal formées
)

selection = []
total = 0
for morceau in morceaux:
    france = morceau[morceau["countries_en"].str.contains("France", na=False)]
    selection.append(france)
    total += len(france)
    if total >= 5000:
        break

df = pd.concat(selection).head(5000)
df.to_csv("data/echantillon_fr.csv", index=False)  # on sauvegarde l'échantillon
print(df.shape)

(5000, 13)


In [4]:
df[["code", "product_name", "brands", "quantity", "categories_en"]].head(10)

,code,product_name,brands,quantity,categories_en
0,0000101209159,Véritable pâte à tartiner noisettes chocolat noir,Bovetti,350 g,"Breakfasts,Spreads,Sweet spreads,fr:pates-a-ta..."
17,0000130008136,Escalope de dinde,NaN,300 g,"Meats and their products,Meats,Poultries,Turke..."
18,0000140323687,Madeleine Framboise,NaN,110 g,"Snacks,Sweet snacks,Biscuits and cakes,Cakes,M..."
19,0000141013129,Croissants margarine,NaN,NaN,"Snacks,Sweet snacks,Sweet pastries and pies,Vi..."
24,0000171812457,Glaces vegetales des alpes,NaN,NaN,NaN
28,0000209024937,The simpsons donuts,NaN,NaN,"Snacks,Sweet snacks,Biscuits and cakes,Cakes,D..."
29,0000228000080,Terrine de campagne,NaN,NaN,"Meats and their products,Meats,Prepared meats,..."
30,0000234022960,Croissants pur beurre,La Fournée Campanière,NaN,"Snacks,Sweet snacks,Sweet pastries and pies,Vi..."
33,0000241013128,Eclairs,Intermarché,180 g,"Snacks,Sweet snacks,Biscuits and cakes,Pastrie..."
34,0000247016406,Beignets gourmands parfum choco-noisettes,NaN,420 g,"Snacks,Sweet snacks,Biscuits and cakes,Cakes,D..."


In [5]:
df.shape


(5000, 13)

In [6]:
(df.isna().mean() * 100).round(1).sort_values(ascending=False)

quantity              51.4
categories_en         29.6
brands                13.7
salt_100g             11.0
saturated-fat_100g     9.6
sugars_100g            8.7
carbohydrates_100g     8.6
fat_100g               8.3
proteins_100g          7.7
energy-kcal_100g       7.6
product_name           2.3
code                   0.0
countries_en           0.0
dtype: float64

In [7]:
df["code"].duplicated().sum()

np.int64(0)

In [8]:
df[["sugars_100g", "carbohydrates_100g", "fat_100g", "salt_100g"]].describe().round(1)

,sugars_100g,carbohydrates_100g,fat_100g,salt_100g
count,4567.0,4570.0,4584.0,4449.0
mean,27.6,47.2,12.8,1.4
std,592.0,593.3,19.4,6.2
min,0.0,0.0,0.0,0.0
25%,2.0,10.0,0.3,0.1
50%,9.6,33.3,5.6,0.5
75%,30.5,64.2,20.5,1.0
max,40000.0,40000.0,533.3,178.6


In [9]:
incoherents = df[df["sugars_100g"] > df["carbohydrates_100g"]]
print(len(incoherents), "produits incohérents")
incoherents[["product_name", "sugars_100g", "carbohydrates_100g"]].head()

10 produits incohérents


,product_name,sugars_100g,carbohydrates_100g
16241,Poppadums,0.800000,0.0413
24613,Seasoning,1.106600,1.0000
29728,"Original Juan, Pain Is Good Jalapeno Pepper Sauce",3.353372,0.0000
47476,Balsamic Vinaigrette Dressing,13.333333,9.1860
53847,Classic Yellow Mustard,0.059852,0.0000
